In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle

from math import log, sqrt
from time import time
from pprint import pprint

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score as AUC, log_loss, accuracy_score as accuracy
from sklearn.metrics import (mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2,
                             explained_variance_score as EVS)
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler, MaxAbsScaler

from keras.models import Sequential
from keras.layers.core import Dense, Dropout
from keras.layers.normalization import BatchNormalization as BatchNorm
from keras.callbacks import EarlyStopping, ModelCheckpoint
from keras.layers.advanced_activations import *
from keras.models import load_model

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

/home/bulent/anaconda3/lib/python3.6/site-packages/h5py/__init__.py:34: FutureWarning: Conversion of the second argument of issubdtype from `float` to `np.floating` is deprecated. In future, it will be treated as `np.float64 == np.dtype(float).type`.
  from ._conv import register_converters as _register_converters
Using TensorFlow backend.


In [2]:
with open('stations-6to31.pkl', 'rb') as f:
    datas = pickle.load(f)
    
x_train6_ = datas['x_train6']
y_train6 = datas['y_train6']
x_train_ = datas['x_train']
y_train = datas['y_train']
x_dev_ = datas['x_dev']
y_dev = datas['y_dev']
x_test_ = datas['x_test']
y_test = datas['y_test']

print(f'x_train shape: {x_train_.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


From the best 21 configurations modes of respective categories are as follows.

**Initializer:** normal

**Layers:** 2

**Batch Size:** 64

**Optimizer:** adamax

**Shuffle:** True

**Scaler:** RobustScaler

**Loss:** mean_absolute_error

In [3]:
def scale_data(scaler, datas):
    # scaler is a scaling function from sklearn library
    # datas is a dictionary, containing 3 sets of x_data with keys - x_train, x_dev, x_test
    # fit on x_train and return the transformed sets of data
    
    x_train = scaler.fit_transform(datas['x_train'].astype(float))
    x_dev = scaler.transform(datas['x_dev'].astype(float))
    x_test = scaler.transform(datas['x_test'].astype(float))
    
    transformed = {'x_train': x_train, 'x_dev': x_dev, 'x_test': x_test}
    return transformed

data6_ = {'x_train': x_train6_, 'x_dev': x_dev_, 'x_test': x_test_}
data_ = {'x_train': x_train_, 'x_dev': x_dev_, 'x_test': x_test_}

data6 = scale_data(RobustScaler(), data6_)
data = scale_data(RobustScaler(), data_)

x_train6 = data6['x_train']
x_train = data['x_train']

x_dev6 = data6['x_dev']
x_dev = data['x_dev']

x_test6 = data6['x_test']
x_test = data['x_test']

print(f'x_train shape: {x_train.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 5), y_train shape: (52416,)
x_train6 shape: (10654, 5), y_train6 shape: (10654,)
x_dev shape: (9011, 5), y_dev shape: (9011,)
x_test shape: (16899, 5), y_test shape: (16899,)


In [4]:
def radstimator(h1=20, h2=15, num_vars=5):
    init = 'normal'
    
    model = Sequential()
    model.add( Dense( h1, kernel_initializer=init, input_dim=num_vars ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( BatchNorm())
    model.add( Dense( h2, kernel_initializer=init ))
    model.add( PReLU( alpha_initializer=init ))
    model.add( Dropout( rate=0.4 ))
    
    model.add( Dense( 1, kernel_initializer=init, activation='linear' ))
    
    return model

In [9]:
print(x_train6_[:5]) # 'Latitude', 'BSH', 'Temperature(avg)', 'Daylength', 'H0'

[[40.141       5.9         4.22916667  9.19499685 13.69287119]
 [40.141       1.3         7.6375      9.20626964 13.74607822]
 [40.141       0.          6.2375      9.21860396 13.80432176]
 [40.141       0.          3.32916667  9.23198817 13.86758193]
 [40.141       0.7         5.06956522  9.24640976 13.93583675]]


In [5]:
data6_3v_ = {'x_train': x_train6_[:, [1, 3, 2, 0]], 'x_dev': x_dev_[:, [1, 3, 2, 0]], 'x_test': x_test_[:, [1, 3, 2, 0]]}
data_3v_ = {'x_train': x_train_[:, [1, 3, 2, 0]], 'x_dev': x_dev_[:, [1, 3, 2, 0]], 'x_test': x_test_[:, [1, 3, 2, 0]]}

data6_3v = scale_data(RobustScaler(), data6_3v_)
data_3v = scale_data(RobustScaler(), data_3v_)

x_train6_3v = data6_3v['x_train']
x_train_3v = data_3v['x_train']

x_dev6_3v = data6_3v['x_dev']
x_dev_3v = data_3v['x_dev']

x_test6_3v = data6_3v['x_test']
x_test_3v = data_3v['x_test']

y_train6_hh0 = y_train6 / x_train6_[:, 4]
y_train_hh0 = y_train / x_train_[:, 4]
y_dev_hh0 = y_dev / x_dev_[:, 4]
y_test_hh0 = y_test / x_test_[:, 4]

print(f'x_train shape: {x_train_3v.shape}, y_train shape: {y_train.shape}')
print(f'x_train6 shape: {x_train6_3v.shape}, y_train6 shape: {y_train6.shape}')
print(f'x_dev shape: {x_dev_3v.shape}, y_dev shape: {y_dev.shape}')
print(f'x_test shape: {x_test_3v.shape}, y_test shape: {y_test.shape}')

x_train shape: (52416, 4), y_train shape: (52416,)
x_train6 shape: (10654, 4), y_train6 shape: (10654,)
x_dev shape: (9011, 4), y_dev shape: (9011,)
x_test shape: (16899, 4), y_test shape: (16899,)


In [6]:
validation_data6 = ( x_dev6_3v, y_dev_hh0 )
for i in range(50):
    rads = radstimator(20, 15, 4)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-4vars-hh0/6stations-hh0-nNTPhi {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train6_3v, y_train6_hh0, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train6_3v, batch_size = 64 )

    mse = MSE( y_train6_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_train6_hh0, p )
    r2 = R2( y_train6_hh0, p )
    evs = EVS( y_train6_hh0, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1,rmse, mae, r2, evs))

C01 » RMSE: 0.1270, MAE: 0.0695, R2: 0.6766, EVS: 0.6769, 
C02 » RMSE: 0.1302, MAE: 0.0663, R2: 0.6602, EVS: 0.6694, 
C03 » RMSE: 0.1313, MAE: 0.0704, R2: 0.6544, EVS: 0.6697, 
C04 » RMSE: 0.1311, MAE: 0.0646, R2: 0.6555, EVS: 0.6592, 
C05 » RMSE: 0.1279, MAE: 0.0679, R2: 0.6722, EVS: 0.6764, 
C06 » RMSE: 0.1287, MAE: 0.0668, R2: 0.6683, EVS: 0.6712, 
C07 » RMSE: 0.1278, MAE: 0.0657, R2: 0.6725, EVS: 0.6748, 
C08 » RMSE: 0.1286, MAE: 0.0686, R2: 0.6686, EVS: 0.6726, 
C09 » RMSE: 0.1277, MAE: 0.0687, R2: 0.6734, EVS: 0.6775, 
C10 » RMSE: 0.1277, MAE: 0.0714, R2: 0.6731, EVS: 0.6757, 
C11 » RMSE: 0.1278, MAE: 0.0656, R2: 0.6728, EVS: 0.6731, 
C12 » RMSE: 0.1281, MAE: 0.0681, R2: 0.6710, EVS: 0.6779, 
C13 » RMSE: 0.1270, MAE: 0.0676, R2: 0.6768, EVS: 0.6797, 
C14 » RMSE: 0.1267, MAE: 0.0691, R2: 0.6785, EVS: 0.6794, 
C15 » RMSE: 0.1275, MAE: 0.0669, R2: 0.6745, EVS: 0.6776, 
C16 » RMSE: 0.1281, MAE: 0.0656, R2: 0.6713, EVS: 0.6797, 
C17 » RMSE: 0.1283, MAE: 0.0670, R2: 0.6700, EVS: 0.6727

In [7]:
validation_data = ( x_dev_3v, y_dev_hh0 )
for i in range(50):
    rads = radstimator(20, 15, 4)
    rads.compile(optimizer='adamax', loss='mean_absolute_error')

    early_stopping = EarlyStopping( monitor = 'val_loss', patience = 10, verbose = 0 )
    filepath = './6to31stats-4vars-hh0/31stations-hh0-nNTPhi {}.h5'.format(i+1)
    checkpointer = ModelCheckpoint(filepath, monitor='val_loss', verbose=0, save_best_only=True )
    history = rads.fit( x_train_3v, y_train_hh0, epochs = 250, batch_size = 64, shuffle = True, 
                         validation_data = validation_data6, callbacks = [ early_stopping, checkpointer ], verbose=0)

    p = rads.predict( x_train_3v, batch_size = 64 )

    mse = MSE( y_train_hh0, p )
    rmse = sqrt( mse )
    mae = MAE( y_train_hh0, p )
    r2 = R2( y_train_hh0, p )
    evs = EVS( y_train_hh0, p )
    
    print('C{:02d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '.format(i+1, rmse, mae, r2, evs))

C01 » RMSE: 0.1062, MAE: 0.0594, R2: 0.7231, EVS: 0.7241, 
C02 » RMSE: 0.1053, MAE: 0.0600, R2: 0.7273, EVS: 0.7293, 
C03 » RMSE: 0.1064, MAE: 0.0599, R2: 0.7220, EVS: 0.7245, 
C04 » RMSE: 0.1064, MAE: 0.0600, R2: 0.7219, EVS: 0.7263, 
C05 » RMSE: 0.1063, MAE: 0.0601, R2: 0.7222, EVS: 0.7240, 
C06 » RMSE: 0.1057, MAE: 0.0593, R2: 0.7255, EVS: 0.7278, 
C07 » RMSE: 0.1066, MAE: 0.0604, R2: 0.7205, EVS: 0.7230, 
C08 » RMSE: 0.1062, MAE: 0.0590, R2: 0.7226, EVS: 0.7242, 
C09 » RMSE: 0.1063, MAE: 0.0601, R2: 0.7225, EVS: 0.7249, 
C10 » RMSE: 0.1056, MAE: 0.0607, R2: 0.7257, EVS: 0.7261, 
C11 » RMSE: 0.1061, MAE: 0.0601, R2: 0.7232, EVS: 0.7236, 
C12 » RMSE: 0.1057, MAE: 0.0583, R2: 0.7253, EVS: 0.7260, 
C13 » RMSE: 0.1053, MAE: 0.0598, R2: 0.7277, EVS: 0.7292, 
C14 » RMSE: 0.1065, MAE: 0.0586, R2: 0.7214, EVS: 0.7220, 
C15 » RMSE: 0.1053, MAE: 0.0588, R2: 0.7274, EVS: 0.7275, 
C16 » RMSE: 0.1057, MAE: 0.0605, R2: 0.7256, EVS: 0.7266, 
C17 » RMSE: 0.1067, MAE: 0.0598, R2: 0.7204, EVS: 0.7225